In [ ]:
# ==============================================================================
# CELL 0: OPUS BOOTSTRAP (SETUP, CONNECTIVITY & AESTHETICS)
# ==============================================================================
# Purpose: 1. Silence warnings for a clean output.
#          2. Mount Google Drive and connect to the Golden Master DB.
#          3. Apply the "Opus Lab" Visual Canon (Standardized Aesthetics).
# ==============================================================================

# --- 1. HYGIENE PROTOCOL ---
import warnings
warnings.simplefilter(action='ignore', category=FutureWarning)
warnings.simplefilter(action='ignore', category=UserWarning)
import pandas as pd
pd.options.mode.chained_assignment = None  # Silence SettingWithCopy
!pip install optuna

# --- 2. IMPORTS ---
import os
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sqlalchemy import create_engine
import sqlite3
from google.colab import drive

# --- 3. CONNECTIVITY ---
print("⏳ Mounting Google Drive...")
try:
    drive.mount('/content/drive')
    print("✅ Drive Mounted.")
except:
    print("ℹ️ Drive already mounted or running locally.")

# DEFINITIVE PATH
DB_PATH = '/content/drive/MyDrive/_Pienza/Assets/Database/opus.db'

if not os.path.exists(DB_PATH):
    print(f"🔴 CRITICAL: Database not found at {DB_PATH}")
else:
    print(f"✅ Database found: {DB_PATH}")
    db_engine = create_engine(f'sqlite:///{DB_PATH}')
    print("✅ SQL Engine Active.")

# --- 4. VISUAL CANON (OPUS LAB THEME) ---
OPUS_PURPLE = '#440154'
OPUS_TEAL   = '#21918c'
OPUS_GREY   = '#FAFAFA'
OPUS_TEXT   = '#121212'

sns.set_theme(style="whitegrid")
plt.rcParams.update({
    'figure.facecolor': OPUS_GREY,
    'axes.facecolor': OPUS_GREY,
    'text.color': OPUS_TEXT,
    'xtick.color': '#333333',
    'ytick.color': '#333333',
    'axes.edgecolor': '#DDDDDD',
    'grid.color': '#E0E0E0',
    'font.family': 'sans-serif',
    'axes.titlecolor': OPUS_PURPLE,
    'axes.titleweight': 'bold',
    'figure.titlesize': 24,
    'figure.titleweight': 'bold'
})

print("✅ Visual Identity Loaded: Opus Lab (Light Mode).")
print("\n--- SYSTEM READY ---")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 413.9/413.9 kB 9.9 MB/s eta 0:00:00
⏳ Mounting Google Drive...
Mounted at /content/drive
✅ Drive Mounted.
✅ Database found: /content/drive/MyDrive/_Pienza/Assets/Database/opus.db
✅ SQL Engine Active.
✅ Visual Identity Loaded: Opus Lab (Light Mode).

--- SYSTEM READY ---


In [ ]:
!pip install -U transformers pysentimiento emoji umap-learn -q
import os
import string
import warnings
import torch
import pandas as pd
import numpy as np
import regex as re
import matplotlib.pyplot as plt
import seaborn as sns
import nltk
import umap

from collections import Counter
from wordcloud import WordCloud, STOPWORDS
from textblob import TextBlob
from nltk.corpus import stopwords
from nltk.stem import PorterStemmer
from nltk.util import ngrams
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from pysentimiento import create_analyzer


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.7/10.7 MB 39.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 608.4/608.4 kB 12.9 MB/s eta 0:00:00


In [ ]:
# ==============================================================================
# CELL 1: RAW OCR EXTRACTION & STRATEGIC GROUPING (JOIN BY NUMERIC SUFFIX)
# ==============================================================================

print("⏳ Forging the Golden Link between OCR and Silver via Substring Join...")

# JOIN inteligente: ignoramos "OCR" (3 chars) y "OF" (2 chars) para unir por el número
query = """
SELECT
    r.ocr_id,
    s.offer_id,
    r.dropoff_address as raw_address,
    s.dropoff_polygon_id,
    s.dropoff_hdbscan_id,
    s.dropoff_polygon_name,
    s.dropoff_hdbscan_name
FROM raw_offers_ocr r
JOIN silver_palette s ON SUBSTR(r.ocr_id, 4) = SUBSTR(s.offer_id, 3)
WHERE r.dropoff_address IS NOT NULL
"""
df_input = pd.read_sql(query, db_engine)

if len(df_input) > 0:
    print(f"✅ Success! Joined {len(df_input)} raw records.")
else:
    print("❌ Critical: Join still returned 0 rows. Please verify if suffixes align (e.g., OCR00001 vs OF00001).")

# --- STRATEGIC GROUPING (Salchichota Logic - Identical as requested) ---
id_map = {
    -1: -1, 41: 0, 42: 0, 46: 0, 43: 1, 65: 2, 62: 2, 44: 2, 36: 2, 49: 3, 52: 3,
    35: 3, 50: 4, 58: 4, 25: 5, 31: 5, 63: 6, 39: 6, 51: 7, 33: 7, 37: 8, 53: 8,
    48: 8, 60: 9, 57: 10, 12: 10, 32: 10, 24: 11, 40: 12, 45: 13, 59: 13, 61: 14,
    38: 14, 34: 15, 30: 16, 66: 16, 17: 17, 14: 17, 22: 17, 16: 18, 13: 18, 11: 19,
    15: 20, 21: 21, 20: 21, 19: 21, 18: 22, 47: 23, 55: 23, 56: 23, 54: 24, 64: 24,
    71: 25, 9: 26, 70: 27, 69: 28, 8: 29, 6: 30, 7: 30, 23: 30, 3: 31, 2: 32,
    4: 33, 29: 33, 68: 34, 5: 35, 27: 36, 28: 36, 1: 37, 10: 38, 0: 39, 26: 40, 67: 41
}

df_input['id_agrupado'] = df_input['dropoff_polygon_id'].fillna(-1).astype(int).map(id_map).fillna(-1)

name_foundry = df_input.groupby('id_agrupado')['dropoff_polygon_name'].unique().apply(
    lambda x: "__".join(sorted([str(i) for i in x if pd.notna(i) and i != 'unassigned']))
).to_dict()
name_foundry[-1] = "unassigned"

df_input['grouped_polyname'] = df_input['id_agrupado'].map(name_foundry)
print("✅ Naming Forge Complete.")

⏳ Forging the Golden Link between OCR and Silver via Substring Join...
✅ Success! Joined 4752 raw records.
✅ Naming Forge Complete.


In [ ]:
# ==============================================================================
# CELL 1.3 + 2: MASTER COALESCE & URBAN STANDARDIZER
# ==============================================================================
# Mission: 1. Finalize the target zone labels with Airport Fusion.
#          2. THEN apply the definitive standardization function.
# ==============================================================================
import regex as re

# --- Definimos la Función ANTES de usarla ---
def standardize_mexico_city_address(text):
    if not isinstance(text, str) or text.lower() in ['null', 'n/a']:
        return "unknown_address"
    t = text.lower()
    t = t.replace('sante fe', 'santa fe').replace('sta fe', 'santa fe').replace('sta fé', 'santa fe').replace('aicm', 'aeropuerto')
    t = re.sub(r'\b(av|ave|avenida|av\.|av_)\b', 'av', t)
    t = re.sub(r'\b(cll|calle|cll\.|cl\.)\b', 'calle', t)
    t = re.sub(r'\b(pº|p de|paseo de|pso|p\.º)\b', 'paseo', t)
    t = re.sub(r'\b(\d{5})\b', r'cp\1', t)
    for _ in range(3):
        t = re.sub(r'(\b\p{L}+(?:_\p{L}+|_\d+)*)\s+(\d{1,4}|norte|sur|este|oeste|poniente|oriente)\b', r'\1_\2', t)
    t = re.sub(r'\b(mexico|méxico|cdmx|ciudad de mexico|df|d\.f\.)\b', '', t)
    t = re.sub(r'[^a-z0-9áéíóúüñ_\s]', ' ', t)
    t = re.sub(r'\s+', ' ', t).strip()
    return t


# --- Ahora ejecutamos el Coalesce ---
print("⏳ Executing Master Coalesce with Airport Fusion...")

# A. Logic for final_zone_id
conditions_id = [(df_input['id_agrupado'] >= 0), (df_input['dropoff_hdbscan_id'] > -1)]
choices_id = ["P_" + df_input['id_agrupado'].astype(int).astype(str),
              "C_" + df_input['dropoff_hdbscan_id'].fillna(-1).astype(int).astype(str)]
df_input['final_zone_id'] = np.select(conditions_id, choices_id, default="Unassigned")

# B. Logic for final_zone_name
conditions_name = [(df_input['id_agrupado'] >= 0), (df_input['dropoff_hdbscan_id'] > -1)]
choices_name = [df_input['grouped_polyname'], df_input['dropoff_hdbscan_name']]
df_input['final_zone_name'] = np.select(conditions_name, choices_name, default="Unassigned Area")

# C. LA FUSIÓN ESTRATÉGICA (AEROPUERTO)
df_input['final_zone_name'] = df_input['final_zone_name'].replace({
    'terminal_1_aicm': 'aicm_aeropuerto',
    'terminal_2_aicm': 'aicm_aeropuerto'
})

# Asumimos que los IDs de HDBSCAN para T1 y T2 son 4 y 5
# (Puedes verificar esto en tu auditoría si falla)
df_input['final_zone_id'] = df_input['final_zone_id'].replace({'C_5': 'C_4'})


# --- Aislamiento del Set de Entrenamiento y Estandarización Final ---
df_nav = df_input[df_input['final_zone_name'] != "Unassigned Area"].copy()
df_nav = df_nav.rename(columns={'raw_address': 'dropoff_address'})

print("⏳ Forging FINAL Standardized Addresses...")
df_nav['clean_address'] = df_nav['dropoff_address'].apply(standardize_mexico_city_address)


print(f"\n✅ Training Set Ready: {len(df_nav)} observations.")
print(f"✅ Airport Fusion: {len(df_nav[df_nav['final_zone_name'] == 'aicm_aeropuerto_total'])} samples for AICM.")
print(f"✅ Final Target Complexity: {df_nav['final_zone_id'].nunique()} unique zones.")

⏳ Executing Master Coalesce with Airport Fusion...
⏳ Forging FINAL Standardized Addresses...

✅ Training Set Ready: 3389 observations.
✅ Airport Fusion: 0 samples for AICM.
✅ Final Target Complexity: 65 unique zones.


In [ ]:
# ==============================================================================
# NLP CONTEXT RESTORATION
# ==============================================================================
import nltk
from nltk.corpus import stopwords
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer

# 1. Asegurar descarga de recursos
try:
    nltk.download('stopwords', quiet=True)
    # Crear la lista bilingüe que el sistema perdió al reiniciar
    stop_words_bilingual = list(set(stopwords.words('spanish') + stopwords.words('english')))
    print(f"✅ Stopwords Bilingües restauradas: {len(stop_words_bilingual)} tokens.")
except Exception as e:
    print(f"❌ Error al restaurar stopwords: {e}")

# 2. Definir target_zones (necesario para los loops de análisis posteriores)
if 'df_nav' in locals():
    target_zones = df_nav['final_zone_name'].unique()
    print(f"✅ Target Zones re-indexadas: {len(target_zones)}")
else:
    print("⚠️ Advertencia: df_nav no detectado. Asegúrate de haber corrido la extracción de datos.")

✅ Stopwords Bilingües restauradas: 504 tokens.
✅ Target Zones re-indexadas: 64


In [ ]:
# ==============================================================================
# CELL 2: THE URBAN STANDARDIZER (Nivel 0) - FINAL RATIFIED VERSION
# ==============================================================================
import regex as re

def standardize_mexico_city_address(text):
    if not isinstance(text, str) or text.lower() in ['null', 'n/a']:
        return "unknown_address"

    t = text.lower()

    # 1. Correcciones Heurísticas
    t = t.replace('sante fe', 'santa fe').replace('sta fe', 'santa fe')
    t = t.replace('sta fé', 'santa fe').replace('aicm', 'aeropuerto')

    # 2. Unificación de Prefijos (Gramática - Se quedan separados)
    t = re.sub(r'\b(av|ave|avenida|av\.|av_)\b', 'av', t)
    t = re.sub(r'\b(cll|calle|cll\.|cl\.)\b', 'calle', t)
    t = re.sub(r'\b(pº|p de|paseo de|pso|p\.º)\b', 'paseo', t)

    # 3. PROTECCIÓN DE CPs (5 dígitos)
    t = re.sub(r'\b(\d{5})\b', r'cp\1', t)

    # 4. FUSIÓN RECURSIVA (La "Soldadura" de Anclaje)
    # Esta regla es potente: busca cualquier palabra o palabra fusionada
    # y la pega al siguiente número o cardinal. Se repite hasta que no haya más.
    for _ in range(3): # 3 pasadas para capturar Eje + 6 + Sur + 123
        t = re.sub(r'(\b\p{L}+(?:_\p{L}+|_\d+)*)\s+(\d{1,4}|norte|sur|este|oeste|poniente|oriente)\b', r'\1_\2', t)

    # 5. Neutralización de Ruido Administrativo
    t = re.sub(r'\b(mexico|méxico|cdmx|ciudad de mexico|df|d\.f\.)\b', '', t)

    # 6. Purificación Unicode Final
    t = re.sub(r'[^a-z0-9áéíóúüñ_\s]', ' ', t)
    t = re.sub(r'\s+', ' ', t).strip()

    return t

print("⏳ Forging FINAL Standardized Addresses...")
df_nav['clean_address'] = df_nav['dropoff_address'].apply(standardize_mexico_city_address)

print("\n--- FINAL FUSION CHECK ---")
samples = [
    "Avenida Insurgentes Sur 1500, San Ángel",
    "Eje 6 Sur 123, Iztapalapa",
    "Circuito Interior Poniente, Miguel Hidalgo",
    "Calle 10 132, Tolteca"
]
for s in samples:
    print(f"IN:  {s}")
    print(f"OUT: {standardize_mexico_city_address(s)}")
    print("-" * 30)

⏳ Forging FINAL Standardized Addresses...

--- FINAL FUSION CHECK ---
IN:  Avenida Insurgentes Sur 1500, San Ángel
OUT: av insurgentes_sur_1500 san ángel
------------------------------
IN:  Eje 6 Sur 123, Iztapalapa
OUT: eje_6 sur_123 iztapalapa
------------------------------
IN:  Circuito Interior Poniente, Miguel Hidalgo
OUT: circuito interior_poniente miguel hidalgo
------------------------------
IN:  Calle 10 132, Tolteca
OUT: calle_10_132 tolteca
------------------------------



¡Esa es una pregunta de Arquitecto de Sistemas de alto nivel! Estás tocando el dilema fundamental entre Ingeniería de Características (Features) y Aprendizaje de Representaciones (Deep Learning).
La respuesta corta: En el mundo real de la producción, se prefiere un "Sanitizador Ligero" antes del Transformer.
Aquí está el porqué desde los Primeros Principios:
1. El Problema de la Escasez de Datos (N ≈ 4,700)
Si tuviéramos 100 millones de direcciones, yo te diría: "No toques nada. Dale el texto crudo y sucio al Transformer; él aprenderá que 'Av', 'ave', 'Avenida' y 'A-v' significan lo mismo basándose en el contexto."
Pero tenemos ~4,700 filas. En el mundo del Deep Learning, eso es un dataset minúsculo.
Sin Regex: El modelo tiene que gastar su limitada capacidad de aprendizaje (sus parámetros) en descubrir que Av. y Avenida son sinónimos.
Con Regex: Tú le "regalas" ese conocimiento. El modelo ya no pierde tiempo en la sintaxis básica y puede usar toda su "atención" en lo que realmente importa: distinguir entre la calle 'Morelos' de Ecatepec y la calle 'Morelos' de Cuajimalpa.
Conclusión: El preprocesamiento (Regex) actúa como un "Multiplicador de Datos". Hace que tus 4,700 ejemplos rindan como si fueran 20,000.
2. El Techo de la Entropía (Señal vs Ruido)
El OCR no solo mete "ruido", mete entropía.
Ejemplo: P.º Reforma vs Paseo de la Reforma.
Para el Transformer, son secuencias de tokens totalmente distintas. Si solo ves P.º cinco veces en todo el año, el modelo no tendrá suficiente evidencia para entender qué es.
Al estandarizar, colapsas 10 variaciones en una sola. Aumentas la densidad de la señal.
3. "¿Qué pasa si ocurre algo nuevo en el futuro?"
Esta es tu preocupación más válida. Si llega un formato nuevo que el Regex no conoce, ¿qué pasa?
La Estrategia del "Sanitizador No Destructivo":
El Regex que diseñamos es defensivo.
Si ve Av., lo cambia a av.
Si ve una palabra nueva que no está en las reglas (ej. un nombre de colonia nuevo o un nuevo formato de calle), el Regex no la toca, simplemente la deja pasar como está.
El Transformer sigue ahí: El Transformer recibirá el texto estandarizado donde conocemos las reglas, y el texto "crudo" donde no las conocemos. Sigue teniendo la capacidad de aprender lo nuevo.
4. El Workflow de Producción Real (The Modern Stack)
En las empresas de vanguardia, el pipeline de inferencia en vivo de Pienza Babel sería:
Capa 1 (Hardware/OCR): Captura de imagen
→
→
 Texto crudo.
Capa 2 (Urban Standardizer): Limpieza de ruido técnico y unificación vial (nuestro Regex v2.0).
Capa 3 (Neural Engine): El Mini-BERT / Transformer procesa el texto limpio y predice la zona.
Capa 4 (Fallback): Si la confianza es baja, se dispara una alerta.
Mi recomendación para Opus:
Mantengamos el Nivel 0 (Regex). No es una muleta, es una guía de precisión. Al limpiar lo "obvio" (prefijos, estados, países), permitimos que la arquitectura del Transformer brille en lo "difícil" (relaciones espaciales y nombres propios).
¿Vientos? Si estás de acuerdo, procedamos con la Celda 3 (Nivel 1: TF-IDF) para ver qué palabras sobreviven a este filtro y se convierten en los verdaderos predictores de zona.

In [ ]:
# ==============================================================================
# CELL 3.1: BoW GLOBAL - IDENTIFYING DOMAIN NOISE
# ==============================================================================
from sklearn.feature_extraction.text import CountVectorizer

print("⏳ Identifying global structural noise via BoW...")

# 1. Corremos un BoW simple para ver qué palabras dominan el dataset completo
bow_vec = CountVectorizer(stop_words=stop_words_bilingual)
X_bow_total = bow_vec.fit_transform(df_nav['clean_address'])
bow_counts = pd.DataFrame({
    'token': bow_vec.get_feature_names_out(),
    'global_freq': X_bow_total.toarray().sum(axis=0)
}).sort_values(by='global_freq', ascending=False)

# 2. Definimos el "Ruido de Dominio" (Top 20 palabras más frecuentes)
# Estas son las que 'ahogan' a las calles específicas.
domain_noise = set(bow_counts.head(25)['token'].tolist())

print(f"✅ Top Global Tokens (Noise Candidates): {list(domain_noise)[:10]}...")

# 3. Actualizamos la lista de Stopwords para el siguiente paso
final_stopwords = list(set(stop_words_bilingual).union(domain_noise))
print(f"✅ Final Stopword list expanded to {len(final_stopwords)} tokens.")

⏳ Identifying global structural noise via BoW...
✅ Top Global Tokens (Noise Candidates): ['chapultepec', 'col', 'miguel', 'cuauhtémoc', 'lomas', 'benito', 'condesa', 'av', 'cuajimalpa', 'morelos']...
✅ Final Stopword list expanded to 529 tokens.


In [ ]:
# ==============================================================================
# CELL 3.1: DYNAMIC SLASH-STOPWORD EXTRACTION (The Scalpel - Fixed)
# ==============================================================================
import regex as re

print("⏳ Executing 'Operation: Slash-Purge' to identify zone labels...")

# 1. Buscamos palabras adyacentes al slash en el texto original
# El regex busca: (Letras) seguido de (espacios opcionales + / + espacios opcionales) seguido de (Letras)
slash_pattern = r'(\p{L}+)\s*/\s*(\p{L}+)'

# FIX: Agregamos .str para que pandas entienda que operamos sobre strings
all_raw_addresses = " ".join(df_nav['dropoff_address'].astype(str).str.lower())
slash_matches = re.findall(slash_pattern, all_raw_addresses)

# Aplanamos la lista de tuplas y convertimos a set para unicidad
slash_noise = set([word for pair in slash_matches for word in pair])

# 2. Audit de lo encontrado
print(f"✅ Identified {len(slash_noise)} words as 'Zone Delimiters'.")
print(f"Sample Slash-Noise: {list(slash_noise)[:15]}...")

# 3. Fusión Final de Stopwords (Bilingüe + BoW Top + Slash Noise)
# Nos aseguramos de que 'domain_noise' esté definido (del intento anterior)
final_stopwords_v2 = list(set(stop_words_bilingual)
                          .union(domain_noise)
                          .union(slash_noise))

print(f"🚀 Final 'Pienza Babel' Stopword list: {len(final_stopwords_v2)} tokens.")

⏳ Executing 'Operation: Slash-Purge' to identify zone labels...
✅ Identified 15 words as 'Zone Delimiters'.
Sample Slash-Noise: ['juaréz', 'doctores', 'colonia', 'satélite', 'cuauhtémoc', 'anzures', 'juárez', 'a', 'lomas', 'n', 'juarez', 's', 'polanco', 'roma', 'condesa']...
🚀 Final 'Pienza Babel' Stopword list: 536 tokens.


In [ ]:
# ==============================================================================
# CELL 3.2: ULTRA-REFINED TF-IDF - THE STREET CORE
# ==============================================================================

print("⏳ Extracting ULTRA-REFINED signals (The True Street Core)...")

tfidf_ultra = TfidfVectorizer(
    ngram_range=(1, 2),
    max_features=2500,
    min_df=2,
    stop_words=final_stopwords_v2
)

X_tfidf_ultra = tfidf_ultra.fit_transform(df_nav['clean_address'])
features_ultra = tfidf_ultra.get_feature_names_out()

ultra_analysis = []

for zone in target_zones:
    mask = (df_nav['final_zone_name'] == zone)
    if mask.sum() == 0: continue

    zone_matrix = X_tfidf_ultra[mask.values]
    mean_weights = zone_matrix.mean(axis=0).A1

    # Top 5 tokens
    top_indices = mean_weights.argsort()[-5:][::-1]
    top_tokens = [features_ultra[i] for i in top_indices]

    ultra_analysis.append({
        'Zone': zone,
        'Obs': mask.sum(),
        'Refined_Street_Signals': ", ".join(top_tokens)
    })

df_ultra = pd.DataFrame(ultra_analysis).sort_values(by='Obs', ascending=False)

print("\n--- PIENZA BABEL: Ultra-Refined Signal Audit ---")
display(df_ultra.head(15))

⏳ Extracting ULTRA-REFINED signals (The True Street Core)...

--- PIENZA BABEL: Ultra-Refined Signal Audit ---


,Zone,Obs,Refined_Street_Signals
17,roma_condesa_2,223,"cp06700, hipódromo, roma_norte, cp06100, roma_..."
19,aicm_aeropuerto,222,"mex, venustiano carranza, carranza, venustiano..."
12,santa_fe_centro_comercial,168,"cp05348, contadero, vasco, contadero cp05348, ..."
6,santa_fe_ibero,150,"cp01219, cp01210, zedec, prol, cp01376"
18,carso_antara_miyana,120,"granada, cp11529, granada cp11529, lago, ejército"
15,santa_fe_quintana__sante_fe_patio,103,"cp01219, cp01376, prol, reforma_400, zedec"
28,herradura_conscripto,99,"herradura, méx, interlomas, cp11200, sotelo cp..."
20,ave_club_de_golf_lomas__interlomas_magnocentro...,96,"interlomas, cp52787, huixquilucan, palmas, cp5..."
10,tamarindos,81,"cp05120, cp05110, tamarindo_90, tamarindo_90 c..."
44,rios,74,"cp06500, río, cp06500 cmx, cuauhtemoc cp06500,..."


In [ ]:
# ==============================================================================
# CELL 4: PIENZA BABEL - WORD EMBEDDINGS (Nivel 2)
# ==============================================================================
# Mission: Train a Word2Vec model to discover the "Semantic Proximity"
#          of Mexico City's addresses.
# ==============================================================================

# --- 1. INSTALACIÓN (Si no está ya presente en el entorno) ---
print("Instalando librerías necesarias (gensim, scikit-learn)...")
!pip install -q gensim scikit-learn # Aseguramos gensim y sklearn
print("Librerías instaladas.")

# --- 2. IMPORTACIONES ---
import torch
import torch.nn as nn
import math
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.manifold import TSNE # Para visualización alternativa (si la necesitas)
from gensim.models import Word2Vec # ¡Ahora sí!
from nltk.corpus import stopwords # Lo importamos por si acaso, aunque no se use aquí
from nltk.tokenize import word_tokenize
from nltk.util import ngrams # Para bigramas/trigramas si los necesitas
import regex as re
from collections import Counter
from google.colab import drive # Para Drive
import os # Para manejo de rutas
import multiprocessing

# --- 3. CONFIGURACIÓN ---
# Aseguramos que los recursos necesarios de NLTK estén descargados
try:
    stopwords.words('spanish')
except LookupError:
    nltk.download('stopwords', quiet=True)
try:
    nltk.data.find('tokenizers/punkt')
except LookupError:
    nltk.download('punkt', quiet=True)

# Montamos Drive si es necesario
if not os.path.exists('/content/drive'):
    drive.mount('/content/drive')

# --- 4. PREPARACIÓN DE DATOS (CARGA Y LIMPIEZA) ---
if 'df_nav' not in locals():
    print("\n🔴 ERROR CRÍTICO: DataFrame 'df_nav' no encontrado. Por favor, ejecuta el Paso 1.3 primero.")
    raise SystemExit("Deteniendo ejecución. Necesitamos los datos limpios.")

# THE FIX: Split each string into a list of tokens
print("⏳ Tokenizing standardized addresses...")
sentences = [str(address).split() for address in df_nav['clean_address'].tolist()]

print(f"\n✅ Corpus ready! ({len(sentences)} documents tokenized).")
# Example check to ensure it worked: [['calle', 'reforma', '222'], ...]
print(f"Sample observation: {sentences[0][:5]}")


# --- 5. ENTRENAMIENTO DEL MODELO WORD2VEC ---
print("\n⏳ Training Pienza Babel's Semantic Engine (Word2Vec)...")

# Parámetros clave para Word2Vec
# - vector_size: La dimensionalidad de los embeddings (64).
# - window: El tamaño de la ventana de contexto (palabras que mira a la vez).
# - min_count: Ignora palabras que aparecen menos de X veces. Muy útil para el ruido.
# - sg=1: Skip-gram. Suele funcionar mejor con datasets pequeños/medianos.
# - workers: Usa todos los cores disponibles para acelerar.
# - seed: Para reproducibilidad.
w2v_model = Word2Vec(
    sentences=sentences,
    vector_size=64,         # Usamos 64 dimensiones, suficiente para capturar algo de semántica
    window=5,               # Ventana de contexto (palabras antes y después)
    min_count=5,            # Ignora palabras que aparecen menos de 5 veces
    sg=1,                   # Skip-gram model
    workers=multiprocessing.cpu_count(), # Usa todos los cores
    seed=42                 # Para reproducibilidad
)

print("✅ Word2Vec Training Complete.")

# --- 6. AUDITORÍA DE PROXIMIDAD SEMÁNTICA (EL PRUEBA DE FUEGO) ---
print("\n--- PIENZA BABEL: Semantic Proximity Audit ---")
test_tokens_semantico = ['reforma', 'calle', 'aeropuerto', 'polanco', 'masaryk', 'insurgentes']

for token in test_tokens_semantico:
    try:
        # Intentamos obtener la similaridad con las palabras más probables
        similares = w2v_model.wv.most_similar(token, topn=5)
        print(f"\n📍 Anchor Token: '{token}'")
    except KeyError:
        print(f"\n📍 Anchor Token: '{token}' (Not found in vocabulary, likely due to min_count or cleaning)")
        continue # Saltamos a la siguiente si no está en el vocabulario

    for t, sim in similares:
        print(f"   --> {t:<25} (Similarity: {sim:.3f})")

print("\n✅ Semantics Learned! Analyzing proximity patterns.")

Instalando librerías necesarias (gensim, scikit-learn)...
Librerías instaladas.
⏳ Tokenizing standardized addresses...

✅ Corpus ready! (3389 documents tokenized).
Sample observation: ['calle', 'barrilaco', 'calle', 'sierra', 'mojada']

⏳ Training Pienza Babel's Semantic Engine (Word2Vec)...
✅ Word2Vec Training Complete.

--- PIENZA BABEL: Semantic Proximity Audit ---

📍 Anchor Token: 'reforma'
   --> paseo                     (Similarity: 0.934)
   --> sierra                    (Similarity: 0.826)
   --> prol                      (Similarity: 0.824)
   --> prolongación              (Similarity: 0.812)
   --> reforma_115               (Similarity: 0.811)

📍 Anchor Token: 'calle'
   --> cp11800                   (Similarity: 0.849)
   --> escandón                  (Similarity: 0.829)
   --> cp11840                   (Similarity: 0.819)
   --> hipódromo                 (Similarity: 0.817)
   --> roma_sur                  (Similarity: 0.817)

📍 Anchor Token: 'aeropuerto'
   --> cp15620   